# Nepal House Price Prediction



In [ ]:
from pathlib import Path
import pickle
import re

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

BASE_DIR = Path.cwd()
RAW_DATA_PATH = BASE_DIR / 'data' / 'kaggle_nepal.csv'
PREPARED_DATA_PATH = BASE_DIR / 'data' / 'nepal_house_data.csv'
BUNDLE_PATH = BASE_DIR / 'model_bundle.pkl'

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


## Data Cleaning And Preparation

In [ ]:
def parse_price(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().replace(',', '')
    if re.search(r'/m|per.?month', s, re.IGNORECASE):
        return np.nan
    s = re.sub(r'Rs\.?\s*', '', s, flags=re.IGNORECASE).strip()
    try:
        if re.search(r'cr', s, re.IGNORECASE):
            return float(re.sub(r'[^\d.]', '', s)) * 10_000_000
        if re.search(r'lakh|lac', s, re.IGNORECASE):
            return float(re.sub(r'[^\d.]', '', s)) * 100_000
        return float(re.sub(r'[^\d.]', '', s))
    except Exception:
        return np.nan


def parse_area(s):
    if pd.isna(s):
        return np.nan
    match = re.search(r'(\d+\.?\d*)', str(s))
    return float(match.group(1)) if match else np.nan


def parse_road(s):
    if pd.isna(s):
        return np.nan
    nums = re.findall(r'(\d+\.?\d*)', str(s))
    if not nums:
        return np.nan
    vals = [float(n) for n in nums]
    return sum(vals) / len(vals)


def parse_location(s):
    if pd.isna(s):
        return np.nan
    return str(s).split(',')[0].strip().title()


def parse_city(s):
    if pd.isna(s):
        return np.nan
    parts = str(s).split(',')
    return parts[-1].strip().title() if len(parts) > 1 else np.nan


CITY_MAP = {
    'kathmandu': 'Kathmandu',
    'kathmandhu': 'Kathmandu',
    'karhmandu': 'Kathmandu',
    'lalitpur': 'Lalitpur',
    'bhaktapur': 'Bhaktapur',
    'sitapaila': 'Kathmandu',
    'narayanthan': 'Kathmandu',
    'rumba chowk': 'Kathmandu',
    'imadol': 'Lalitpur',
}


def normalise_city(s):
    if pd.isna(s):
        return 'Other'
    return CITY_MAP.get(str(s).strip().lower(), str(s).strip().title())


df = pd.read_csv(RAW_DATA_PATH)
print(f'Raw rows loaded: {len(df)}')

df['Price_NPR'] = df['PRICE'].apply(parse_price)
df['Area_Anna'] = df['LAND AREA'].apply(parse_area)
df['Road_Width_Ft'] = df['ROAD ACCESS'].apply(parse_road)
df['Location'] = df['LOCATION'].apply(parse_location)
df['City'] = df['LOCATION'].apply(parse_city).apply(normalise_city)
df['District'] = df['City']
df['BHK'] = pd.to_numeric(df['BEDROOM'], errors='coerce')
df['Bathrooms'] = pd.to_numeric(df['BATHROOM'], errors='coerce')
df['Floors'] = pd.to_numeric(df['FLOOR'], errors='coerce')

df = df[df['TITLE'].str.contains('sale', case=False, na=False)]
print(f'After sale filter: {len(df)} rows')

clean = df[['City', 'District', 'Location', 'BHK', 'Bathrooms', 'Floors', 'Area_Anna', 'Road_Width_Ft', 'Price_NPR']].copy()

for col in ['BHK', 'Bathrooms', 'Floors']:
    loc_median = clean.groupby('Location')[col].transform('median')
    global_median = clean[col].median()
    clean[col] = clean[col].fillna(loc_median).fillna(global_median)

loc_road = clean.groupby('Location')['Road_Width_Ft'].transform('median')
clean['Road_Width_Ft'] = clean['Road_Width_Ft'].fillna(loc_road).fillna(12.0)

before = len(clean)
clean.dropna(subset=['Price_NPR', 'Area_Anna'], inplace=True)
clean = clean[clean['Bathrooms'] <= clean['BHK'] + 3]
clean = clean[(clean['Price_NPR'] >= 1_000_000) & (clean['Price_NPR'] <= 500_000_000)]
clean = clean[(clean['Area_Anna'] >= 0.5) & (clean['Area_Anna'] <= 50)]
clean = clean[(clean['BHK'] >= 1) & (clean['BHK'] <= 15)]
clean = clean[(clean['Road_Width_Ft'] >= 4) & (clean['Road_Width_Ft'] <= 100)]
clean = clean.reset_index(drop=True)

after = len(clean)
print(f'Clean rows: {after} (removed {before - after} rows after imputation)')
print(f"Price range: NPR {clean['Price_NPR'].min():,.0f} - {clean['Price_NPR'].max():,.0f}")
print(f"Cities: {sorted(clean['City'].unique())}")
print(f"Locations: {clean['Location'].nunique()} unique")

PREPARED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(PREPARED_DATA_PATH, index=False)
print(f'Saved cleaned data to {PREPARED_DATA_PATH}')

clean.head()


## Model Training And Evaluation

In [ ]:
data = clean.copy()

RARE_THRESHOLD = 10
RATE_SUPPORT_THRESHOLD = 3
RATE_SMOOTHING = 5

data = data[data['Bathrooms'] <= data['BHK'] + 3]
q_low = data['Price_NPR'].quantile(0.01)
q_high = data['Price_NPR'].quantile(0.99)
data = data[(data['Price_NPR'] >= q_low) & (data['Price_NPR'] <= q_high)]
print(f'After cleaning: {len(data)} rows')

data['City'] = data['City'].apply(
    lambda x: CITY_MAP.get(str(x).strip().lower(), str(x).strip().title()) if pd.notna(x) else 'Other'
)
data['Raw_Location'] = data['Location'].apply(
    lambda x: str(x).strip().title() if pd.notna(x) and str(x).strip() else 'Other'
)

loc_counts = data['Raw_Location'].value_counts()
rare_locs = loc_counts[loc_counts < RARE_THRESHOLD].index
data['Location'] = data['Raw_Location'].apply(lambda x: 'Other' if x in rare_locs else x)
print(f"Locations after grouping: {data['Location'].nunique()}")

data['price_per_anna_raw'] = data['Price_NPR'] / data['Area_Anna']
ppa_q99 = data['price_per_anna_raw'].quantile(0.99)
data = data[data['price_per_anna_raw'] <= ppa_q99]

loc_rate = data.groupby('Location')['price_per_anna_raw'].median()
city_rate = data.groupby('City')['price_per_anna_raw'].median()
global_rate = data['price_per_anna_raw'].median()

raw_loc_counts = data['Raw_Location'].value_counts()
raw_loc_rate = data.groupby('Raw_Location')['price_per_anna_raw'].median()
raw_loc_city = data.groupby('Raw_Location')['City'].agg(
    lambda s: s.mode().iloc[0] if not s.mode().empty else 'Other'
)

smoothed_loc_rate = {}
for loc, rate in raw_loc_rate.items():
    loc_count = int(raw_loc_counts.get(loc, 0))
    loc_city_name = raw_loc_city.get(loc, 'Other')
    prior_rate = float(city_rate.get(loc_city_name, global_rate))
    smoothed_loc_rate[loc] = float(((loc_count * float(rate)) + (RATE_SMOOTHING * prior_rate)) / (loc_count + RATE_SMOOTHING))

data['loc_land_rate'] = data['Raw_Location'].map(smoothed_loc_rate).fillna(data['City'].map(city_rate)).fillna(global_rate)
data['city_land_rate'] = data['City'].map(city_rate).fillna(global_rate)
print(f'After per-anna outlier removal: {len(data)} rows')

data['BHK_per_Anna'] = data['BHK'] / data['Area_Anna']
data['Bath_per_BHK'] = data['Bathrooms'] / data['BHK']
data['Area_x_Floors'] = data['Area_Anna'] * data['Floors']
data['Road_x_Area'] = data['Road_Width_Ft'] * data['Area_Anna']
data['Log_Price'] = np.log(data['Price_NPR'])

data_encoded = pd.get_dummies(data, columns=['Location', 'City'])
drop_cols = ['Price_NPR', 'Log_Price', 'District', 'price_per_anna_raw', 'Raw_Location']
X = data_encoded.drop(columns=[c for c in drop_cols if c in data_encoded.columns])
y = data['Log_Price']
print(f'Feature count: {X.shape[1]}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} rows | Holdout: {len(X_test)} rows')

model_columns = list(X.columns)
meta = {
    'log_transformed': True,
    'rate_maps': {
        'loc_rate': smoothed_loc_rate,
        'city_rate': city_rate.to_dict(),
        'global_rate': global_rate,
    },
    'resolver_locations': sorted(
        loc for loc, count in raw_loc_counts.items()
        if loc != 'Other' and int(count) >= RATE_SUPPORT_THRESHOLD
    ),
}

model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    max_features=0.6,
    subsample=0.8,
    random_state=42,
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')
print('K-fold R2 scores:', [round(s, 3) for s in cv_r2])
print(f'K-fold mean R2: {cv_r2.mean():.3f} +- {cv_r2.std():.3f}')

model.fit(X_train, y_train)
y_pred = np.exp(model.predict(X_test))
y_actual = np.exp(y_test.values)
mae = mean_absolute_error(y_actual, y_pred)
r2 = r2_score(y_actual, y_pred)

print(f'Holdout MAE: NPR {mae:,.0f}')
print(f'Holdout R2: {r2:.3f}')

importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = importance.sort_values(ascending=False).head(10)
display(top_features.to_frame('importance'))

model.fit(X, y)

with open(BUNDLE_PATH, 'wb') as f:
    pickle.dump({'model': model, 'model_columns': model_columns, 'meta': meta}, f)

with open(BASE_DIR / 'house_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open(BASE_DIR / 'model_columns.pkl', 'wb') as f:
    pickle.dump(model_columns, f)
with open(BASE_DIR / 'model_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print(f'Saved model bundle to {BUNDLE_PATH}')


## Example Prediction

In [ ]:
log_transformed = meta.get('log_transformed', False)
rate_maps = meta.get('rate_maps', {})
loc_rate = rate_maps.get('loc_rate', {})
city_rate = rate_maps.get('city_rate', {})
global_rate = rate_maps.get('global_rate', 8_000_000)


def normalize_city(city_input):
    return CITY_MAP.get(city_input.strip().lower(), city_input.strip().title())


def get_land_rates(resolved_location, city_input):
    city_norm = normalize_city(city_input)
    land_rate = loc_rate.get(resolved_location) or city_rate.get(city_norm) or global_rate
    c_rate = city_rate.get(city_norm, global_rate)
    return float(land_rate), float(c_rate)


def build_input(area, bhk, bath, floors, road, resolved, city_input):
    input_df = pd.DataFrame(0, index=[0], columns=model_columns)
    input_df['Area_Anna'] = area
    input_df['BHK'] = bhk
    input_df['Bathrooms'] = bath
    input_df['Floors'] = floors
    input_df['Road_Width_Ft'] = road
    input_df['BHK_per_Anna'] = bhk / area if area > 0 else 0
    input_df['Bath_per_BHK'] = bath / bhk if bhk > 0 else 0
    input_df['Area_x_Floors'] = area * floors
    input_df['Road_x_Area'] = road * area

    land_rate, c_rate = get_land_rates(resolved or '', city_input)
    if 'loc_land_rate' in input_df.columns:
        input_df['loc_land_rate'] = land_rate
    if 'city_land_rate' in input_df.columns:
        input_df['city_land_rate'] = c_rate

    city_col = 'City_' + normalize_city(city_input)
    if city_col in input_df.columns:
        input_df[city_col] = 1
    elif 'City_Other' in input_df.columns:
        input_df['City_Other'] = 1

    if resolved:
        loc_col = 'Location_' + resolved
        if loc_col in input_df.columns:
            input_df[loc_col] = 1

    return input_df


def fmt(val):
    if val >= 10_000_000:
        return f'{val / 10_000_000:.2f} Crore'
    if val >= 100_000:
        return f'{val / 100_000:.2f} Lakh'
    return f'NPR {val:,.0f}'


available_locations = sorted(col.replace('Location_', '') for col in model_columns if col.startswith('Location_'))
sample_location = next(
    (loc for loc in ['Baneshwor', 'Naxal', 'Maharajgunj', 'Budhanilkantha'] if loc in available_locations),
    available_locations[0] if available_locations else None,
)

sample = {
    'location': sample_location,
    'city': 'Kathmandu',
    'area': 4.0,
    'bhk': 3,
    'bath': 2,
    'floors': 2.5,
    'road': 13.0,
}

sample_input = build_input(
    area=sample['area'],
    bhk=sample['bhk'],
    bath=sample['bath'],
    floors=sample['floors'],
    road=sample['road'],
    resolved=sample['location'],
    city_input=sample['city'],
)

raw_prediction = float(model.predict(sample_input)[0])
prediction = float(np.exp(raw_prediction)) if log_transformed else raw_prediction
low = prediction * 0.88
high = prediction * 1.12

print('Sample input:', sample)
print(f"Predicted price: NPR {prediction:,.0f}")
print(f"Formatted price: {fmt(prediction)}")
print(f"Estimated range: {fmt(low)} to {fmt(high)}")
